In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.svm import SVR

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

import mlflow

/home/kibria/miniconda3/envs/Uber/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# set the dagshub tracking server

mlflow.set_tracking_uri("https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow")

In [4]:
import dagshub
dagshub.init(repo_owner='gulamkibria775', repo_name='Uber-Demand-Prediction', mlflow=True)

Accessing as gulamkibria775

Initialized MLflow to track repo "gulamkibria775/Uber-Demand-Prediction"

Repository gulamkibria775/Uber-Demand-Prediction initialized!

In [5]:
# load the training and test data

train_data_path = "../data/processed/train.csv"
test_data_path = "../data/processed/test.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

train_df

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [6]:
# missing value in training data

train_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [7]:
# missing values in the test data

test_df.isna().sum()

lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

In [8]:
# make X_train and y_train

X_train = train_df.drop(columns=["total_pickups"])

y_train = train_df["total_pickups"]

In [9]:
X_train.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4


In [10]:
# make X_test and y_test

X_test = test_df.drop(columns=["total_pickups"])

y_test = test_df["total_pickups"]

In [11]:
X_test.head()

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,38.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,39.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,37.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,41.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,38.0,1


In [12]:
from sklearn import set_config

set_config(transform_output="pandas")

In [13]:
# encode the data

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first",sparse_output=False), ["region","day_of_week"])
], remainder="passthrough", n_jobs=-1,force_int_remainder_cols=False)

In [14]:
encoder

,transformers,"[('ohe', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,-1
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,False
,categories,'auto'
,drop,'first'
,sparse_output,False


In [15]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

/home/kibria/miniconda3/envs/Uber/lib/python3.10/site-packages/sklearn/compose/_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


In [17]:
import optuna
import tqdm 

In [18]:
# set the experiment

mlflow.set_experiment("Model Selection")

2026/04/20 23:21:22 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/6da9ee55a62d4893989a556d49b2d0ac', creation_time=1776705683068, experiment_id='1', last_update_time=1776705683068, lifecycle_stage='active', name='Model Selection', tags={}, trace_location=None, workspace='default'>

In [19]:
def objective(trial):
    # start the child run
    with mlflow.start_run(nested=True) as child:
        
        # model name search space
        list_of_models = ["LR", "RF", "GBR", "XGBR"]
        model_name = trial.suggest_categorical("model_name", list_of_models)
    
        if model_name == "LR":
            model = LinearRegression()
    
        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,100,step=10)
            max_depth_rf = trial.suggest_int("max_depth_rf",3,10)
            model = RandomForestRegressor(n_estimators=n_estimators_rf, 
                                          max_depth=max_depth_rf, 
                                          random_state=42, n_jobs=-1)
    
        elif model_name == "GBR":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,100,step=10)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",1e-4,1e-1, log=True)
            model = GradientBoostingRegressor(n_estimators=n_estimators_gb, 
                                              learning_rate=learning_rate_gb,
                                             random_state=42)
    
        elif model_name == "XGBR":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,100,step=10)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",1e-4,1e-1, log=True)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",3,10)
            model = XGBRegressor(n_estimators=n_estimators_xgb,
                                learning_rate=learning_rate_xgb,
                                max_depth=max_depth_xgb)
    
        # log the model name
        mlflow.log_param("model_name",model_name)
        
        # log the model parameters
        mlflow.log_params(model.get_params())
        
        # fit on the data
        model.fit(X_train_encoded,y_train)
    
        # get the predictions
        y_pred = model.predict(X_test_encoded)
    
        # calculate the loss
        loss = mean_absolute_percentage_error(y_test, y_pred)
    
        # log the metric
        mlflow.log_metric("MAPE",loss)
        return loss

In [20]:
# optimize the objective function

with mlflow.start_run(run_name="best_model", nested=True) as parent:

    # create a study object
    study = optuna.create_study(study_name="model_selection", direction="minimize")
    # optimize the objective function
    study.optimize(func=objective, n_trials=50, n_jobs=-1)
    
    # log the best parameters
    mlflow.log_params(study.best_params)
    # log the best error value
    mlflow.log_metric("Best_MAPE", study.best_value)

[I 2026-04-20 23:22:32,651] A new study created in memory with name: model_selection


🏃 View run loud-lynx-546 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/16ca2dcd1f6942d8aabb1ec3daa87752
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1


[I 2026-04-20 23:23:06,069] Trial 0 finished with value: 5.9415602684021 and parameters: {'model_name': 'XGBR', 'n_estimators_xgb': 10, 'learning_rate_xgb': 0.010395578277685112, 'max_depth_xgb': 7}. Best is trial 0 with value: 5.9415602684021.


🏃 View run calm-sow-809 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/23528159d846432285c4016c82fc155d
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1
🏃 View run luxuriant-gull-350 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/f548aab39b9045088dd4907742af6c71
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1
🏃 View run colorful-lamb-60 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/fac6d95ef03343568292595d2b2505a7
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1
🏃 View run agreeable-worm-930 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/7dbae13959bf47dca7706b136cb61da1
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/

[I 2026-04-20 23:23:26,995] Trial 6 finished with value: 0.29003821492025283 and parameters: {'model_name': 'RF', 'n_estimators_rf': 80, 'max_depth_rf': 10}. Best is trial 6 with value: 0.29003821492025283.


...

🏃 View run whimsical-cod-417 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/5ac60ecc472e41499ab9d6b6bed52029
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1


[I 2026-04-20 23:27:19,516] Trial 5 finished with value: 5.87486468268694 and parameters: {'model_name': 'GBR', 'n_estimators_gb': 40, 'learning_rate_gb': 0.0030120266640552974}. Best is trial 22 with value: 0.28987187209242254.


🏃 View run able-hen-139 at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/9853c48f6a544aa281d6dbd8d13e270d
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1


[I 2026-04-20 23:27:22,593] Trial 34 finished with value: 0.2927120884443459 and parameters: {'model_name': 'RF', 'n_estimators_rf': 40, 'max_depth_rf': 9}. Best is trial 22 with value: 0.28987187209242254.


🏃 View run best_model at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1/runs/f01b767bbdd84c149c105792a7465c9a
🧪 View experiment at: https://dagshub.com/gulamkibria775/Uber-Demand-Prediction.mlflow/#/experiments/1


In [21]:
# best value

study.best_value

0.28987187209242254

In [22]:
# best parameters

study.best_params

{'model_name': 'RF', 'n_estimators_rf': 30, 'max_depth_rf': 10}

In [23]:
# model value counts

study.trials_dataframe()['params_model_name'].value_counts()

params_model_name
RF      34
XGBR     6
GBR      6
LR       4
Name: count, dtype: int64

In [24]:
from optuna.visualization import (
    plot_optimization_history, 
    plot_parallel_coordinate, 
    plot_param_importances
)

In [27]:
plot_optimization_history(study)

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [28]:
plot_parallel_coordinate(study, params=["model_name"])

ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

In [29]:
# train the linear regression model

lr = LinearRegression()

lr.fit(X_train_encoded, y_train)

# get predictions
y_pred_train = lr.predict(X_train_encoded) 
y_pred_test = lr.predict(X_test_encoded)

# loss

mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
mape_test = mean_absolute_percentage_error(y_test, y_pred_test)

print("The training error is ", mape_train)
print("The test error is ", mape_test)

The training error is  0.31749731735894027
The test error is  0.3069871006888052


In [30]:
lr.coef_

array([ 6.59266116, -2.05638952,  1.61048142,  3.5589809 ,  9.08819126,
        2.48639847,  7.8520678 , 10.1342072 , -1.11046656,  8.27602693,
        5.45456779, 10.55872532, -1.47820493,  7.18907985,  6.87749488,
       -1.37837448, -1.70783813, 13.33847201,  5.6738974 ,  3.55876455,
       11.32924195,  5.90374261,  2.88449163, -2.03676138,  2.83674402,
        2.42578855,  6.85780565, -1.94162399, -1.62963472,  0.64633354,
        0.93102079,  1.43743235,  1.61026612,  0.91215046, -0.468891  ,
        1.40961414,  0.49144781,  0.28244361,  0.18601736, -1.42136691])

In [31]:
def tune_ridge(trial):
    # hyperparameter space
    alpha = trial.suggest_float("alpha",30,100)
    
    # make the model object
    ridge = Ridge(alpha=alpha, random_state=42)
    
    # train the model
    ridge.fit(X_train_encoded, y_train)
    
    # get predictions
    y_pred = ridge.predict(X_test_encoded)
    
    # calculate loss
    loss = mean_absolute_percentage_error(y_test, y_pred)

    return loss
        

In [32]:
# create study

study = optuna.create_study(study_name="tune_model", direction="minimize")

[I 2026-04-20 23:30:28,061] A new study created in memory with name: tune_model


In [33]:
# optimize

study.optimize(func=tune_ridge, n_trials=100, n_jobs=-1, show_progress_bar=True)

Best trial: 4. Best value: 0.306878:  11%|█         | 11/100 [00:00<00:04, 19.82it/s]

[I 2026-04-20 23:30:41,542] Trial 2 finished with value: 0.3070367673853913 and parameters: {'alpha': 96.37192152287254}. Best is trial 2 with value: 0.3070367673853913.
[I 2026-04-20 23:30:41,551] Trial 7 finished with value: 0.30688004049479756 and parameters: {'alpha': 43.798427937707906}. Best is trial 7 with value: 0.30688004049479756.
[I 2026-04-20 23:30:41,556] Trial 5 finished with value: 0.30693534256053917 and parameters: {'alpha': 71.33323690591902}. Best is trial 7 with value: 0.30688004049479756.
[I 2026-04-20 23:30:41,560] Trial 4 finished with value: 0.30687806474513757 and parameters: {'alpha': 37.53345357074372}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,568] Trial 1 finished with value: 0.30695124910766186 and parameters: {'alpha': 76.0298659145573}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,573] Trial 14 finished with value: 0.3069142237544348 and parameters: {'alpha': 64.06867644146271}. Best is trial 4 wit

Best trial: 4. Best value: 0.306878:  19%|█▉        | 19/100 [00:00<00:02, 30.55it/s]

[I 2026-04-20 23:30:41,579] Trial 3 finished with value: 0.3069497701841342 and parameters: {'alpha': 75.61312982497543}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,580] Trial 15 finished with value: 0.30691732912990716 and parameters: {'alpha': 65.22605120504977}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,586] Trial 10 finished with value: 0.30688164731965323 and parameters: {'alpha': 30.667656180844173}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,588] Trial 12 finished with value: 0.30692498165864557 and parameters: {'alpha': 67.94108230007137}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,682] Trial 16 finished with value: 0.30688162289467713 and parameters: {'alpha': 45.89995607817602}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,738] Trial 19 finished with value: 0.3068834549164627 and parameters: {'alpha': 47.82451473555373}. Best is trial

[I 2026-04-20 23:30:41,745] Trial 18 finished with value: 0.3068837928468662 and parameters: {'alpha': 48.139751612599774}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,807] Trial 22 finished with value: 0.3068796621486066 and parameters: {'alpha': 33.080515471829116}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,841] Trial 21 finished with value: 0.3068845256753839 and parameters: {'alpha': 48.79295089198679}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,919] Trial 25 finished with value: 0.306879317387977 and parameters: {'alpha': 33.629406610655835}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,928] Trial 24 finished with value: 0.30688047169776905 and parameters: {'alpha': 31.9762221430303}. Best is trial 4 with value: 0.30687806474513757.
[I 2026-04-20 23:30:41,929] Trial 23 finished with value: 0.3068810394869188 and parameters: {'alpha': 31.316429174976342}. Best is trial 

...

[I 2026-04-20 23:30:43,132] Trial 68 finished with value: 0.30696193919489306 and parameters: {'alpha': 78.9521411918003}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,163] Trial 71 finished with value: 0.3069440012401982 and parameters: {'alpha': 73.95226617194449}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,194] Trial 72 finished with value: 0.30687805243237254 and parameters: {'alpha': 37.7588860462585}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,261] Trial 73 finished with value: 0.3068791554683081 and parameters: {'alpha': 33.92273594262796}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,261] Trial 74 finished with value: 0.3068780613038146 and parameters: {'alpha': 38.34253283457754}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,272] Trial 76 finished with value: 0.30687897236721995 and parameters: {'alpha': 34.2775348516895}. Best is tria

Best trial: 30. Best value: 0.306878:  89%|████████▉ | 89/100 [00:02<00:00, 45.49it/s]

[I 2026-04-20 23:30:43,430] Trial 79 finished with value: 0.3068790008401904 and parameters: {'alpha': 34.220629951854264}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,434] Trial 81 finished with value: 0.3068781552990544 and parameters: {'alpha': 39.12607977717148}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,435] Trial 80 finished with value: 0.30687807583554816 and parameters: {'alpha': 38.52833306060758}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,488] Trial 84 finished with value: 0.30688217837359 and parameters: {'alpha': 30.139421167639483}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,493] Trial 82 finished with value: 0.30688223519294217 and parameters: {'alpha': 30.084682553349815}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,495] Trial 83 finished with value: 0.3068781656543662 and parameters: {'alpha': 39.189235414859965}. Best is 

Best trial: 30. Best value: 0.306878: 100%|██████████| 100/100 [00:02<00:00, 37.49it/s]

[I 2026-04-20 23:30:43,665] Trial 88 finished with value: 0.306878050637173 and parameters: {'alpha': 38.09025775151626}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,765] Trial 91 finished with value: 0.3068780488946229 and parameters: {'alpha': 37.95530648108646}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,779] Trial 92 finished with value: 0.3068780547760143 and parameters: {'alpha': 37.701109366234235}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,793] Trial 93 finished with value: 0.3068780577533894 and parameters: {'alpha': 37.64526750793011}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,832] Trial 95 finished with value: 0.3068788797082771 and parameters: {'alpha': 41.60852944448773}. Best is trial 30 with value: 0.30687804873641195.
[I 2026-04-20 23:30:43,834] Trial 94 finished with value: 0.3068786027082026 and parameters: {'alpha': 40.8819001998138}. Best is trial

In [34]:
# best parameters

study.best_params

{'alpha': 37.995407150873625}

In [35]:
# best value

study.best_value

0.30687804873641195